# Generation statistics — CO2RR-to-CO Stage A (w=1.0 / w=2.5)

Analyses the guided-generation pool and the cheap-screen funnel for the **economic
CO2RR-to-CO** run (sibling of `15_generation_stats.ipynb`, which is the HER version).

Inputs (from `scripts/21_generate_co2rr_economic.sh` + `scripts/30_cheap_screen_co2rr.py`):
- `outputs/gen_co2rr_economic/w1p0|w2p5/chunk_*/*.cif` — raw generated carbides
- `outputs/gen_co2rr_economic/top_CO.csv` — cheap-screen survivors (economic HEC, CO-releasing)

CO2RR descriptors are recomputed from each CIF (self-contained). Design targets:
carbide, CO-release `|best_site_co_affinity|<0.20`, economic `expensive_metal_fraction==0`,
HEC `n_metal>=3` & `S_mix>=1.0`. **Re-runnable** — analyses whatever chunks exist.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

ROOT = Path('/home/jonglee69/mattergen/CarbideMatterGen'); sys.path.insert(0, str(ROOT))
GEN = ROOT/'outputs'/'gen_co2rr_economic'; FIG = ROOT/'figures'; FIG.mkdir(exist_ok=True)
CO_OPT = 0.0; CO_WINDOW = 0.20
NATURE_RC = {'font.family':'sans-serif','font.sans-serif':['Arial','DejaVu Sans'],
    'font.size':8,'axes.labelsize':8,'axes.titlesize':9,'xtick.labelsize':7,'ytick.labelsize':7,
    'legend.fontsize':7,'axes.linewidth':0.6,'xtick.direction':'in','ytick.direction':'in',
    'legend.frameon':False,'pdf.fonttype':42,'savefig.bbox':'tight','savefig.dpi':300}
mpl.rcParams.update(NATURE_RC)
OI = {'blue':'#0072B2','vermil':'#D55E00','green':'#009E73','orange':'#E69F00','sky':'#56B4E9','purple':'#CC79A7'}
print('GEN dir:', GEN)


## 1. Load pools and recompute CO2RR descriptors


In [ ]:
from pymatgen.core import Structure
from carbidemattergen.labeling import is_tm_carbide_composition
from carbidemattergen.co2rr import co2rr_descriptors
from carbidemattergen.hydrogen import carbon_fraction, expensive_metal_fraction
from carbidemattergen.extra_descriptors import metal_mixing_entropy, n_metal_species

def load_pool(weight_tag):
    files = sorted((GEN/weight_tag).rglob('*.cif'))
    rows = []
    for fp in files:
        try: comp = Structure.from_file(str(fp)).composition
        except Exception: continue
        co = co2rr_descriptors(comp)
        rows.append({'w': weight_tag, 'formula': comp.reduced_formula,
            'is_carbide': is_tm_carbide_composition(comp),
            'best_co': co['best_site_co_affinity'], 'mean_co': co['mean_co_affinity'],
            'frac_co_sel': co['frac_co_selective_sites'], 'cooh': co['mean_cooh_affinity'],
            'co2rr_sel': co['co2rr_selectivity'], 'xC': carbon_fraction(comp),
            'expensive': expensive_metal_fraction(comp),
            'S_mix': metal_mixing_entropy(comp), 'n_metal': n_metal_species(comp)})
    df = pd.DataFrame(rows)
    nc = int(df.is_carbide.sum()) if len(df) else 0
    print(f'{weight_tag}: {len(files)} CIFs -> {len(df)} parsed, {nc} carbides')
    return df

pools = {tag: load_pool(tag) for tag in ('w1p0','w2p5') if (GEN/tag).exists()}
df_all = pd.concat(pools.values(), ignore_index=True) if pools else pd.DataFrame()
print('combined:', len(df_all), 'structures')


## 2. Numerical summary (carbide subset)


In [ ]:
def summarise(df):
    d = df[df.is_carbide]
    rows = [('n_carbide', len(d), '', '')]
    for col, lab in [('best_co','best-site dG_CO'),('mean_co','mean dG_CO'),
                     ('frac_co_sel','frac CO-selective'),('cooh','mean dG_COOH'),
                     ('co2rr_sel','CO2RR selectivity'),('S_mix','S_mix (R)'),
                     ('n_metal','n_metal'),('expensive','expensive frac'),('xC','carbon frac')]:
        if col in d.columns and len(d):
            s = d[col].dropna()
            if len(s): rows.append((lab, f'{s.mean():.3f}', f'{s.min():.3f}', f'{s.max():.3f}'))
    return pd.DataFrame(rows, columns=['metric','mean','min','max'])
for tag, df in pools.items():
    print(f'=== {tag} (carbide subset) ==='); print(summarise(df).to_string(index=False)); print()


## 3. Six-panel CO2RR generation-statistics figure

Carbide subset: (a) best-site $\Delta G_{CO}$ with CO-release window; (b) mean vs best $\Delta G_{CO}$;
(c) entropy vs best $\Delta G_{CO}$; (d) metal-count histogram; (e) economic (expensive-metal fraction);
(f) CO2RR selectivity vs best $\Delta G_{CO}$.


In [ ]:
d = df_all[df_all.is_carbide].copy() if len(df_all) else pd.DataFrame()
fig, ax = plt.subplots(2, 3, figsize=(7.8, 5.0))
for a, ch in zip(ax.flat, 'abcdef'):
    a.text(-0.18, 1.06, ch, transform=a.transAxes, fontsize=10, fontweight='bold', va='top')
if len(d):
    ax[0,0].hist(d.best_co.dropna(), bins=np.linspace(-1.0,0.4,40), color=OI['blue'], edgecolor='black', linewidth=0.3, alpha=0.85)
    ax[0,0].axvspan(CO_OPT-CO_WINDOW, CO_OPT+CO_WINDOW, color=OI['vermil'], alpha=0.18, label='CO window')
    ax[0,0].axvline(0, color='gray', ls=':', lw=0.6)
    ax[0,0].set_xlabel(r'best-site $\Delta G_{CO}$ (eV)'); ax[0,0].set_ylabel('count'); ax[0,0].legend(loc='upper left', fontsize=6)
    ax[0,1].hexbin(d.mean_co, d.best_co, gridsize=35, mincnt=1, cmap='Blues', norm=LogNorm())
    ax[0,1].axhline(0, color=OI['vermil'], ls=':', lw=0.6); ax[0,1].set_xlabel(r'mean $\Delta G_{CO}$ (eV)'); ax[0,1].set_ylabel(r'best-site $\Delta G_{CO}$ (eV)')
    ax[0,2].hexbin(d.S_mix, d.best_co, gridsize=35, mincnt=1, cmap='BuGn', norm=LogNorm())
    ax[0,2].axvline(1.0, color='gray', ls=':', lw=0.6, label='HEA 1.0R'); ax[0,2].axhline(0, color=OI['vermil'], ls=':', lw=0.6)
    ax[0,2].set_xlabel(r'$S_{mix}/R$'); ax[0,2].set_ylabel(r'best-site $\Delta G_{CO}$ (eV)'); ax[0,2].legend(loc='upper right', fontsize=6)
    nm = d.n_metal.dropna().astype(int)
    ax[1,0].hist(nm, bins=np.arange(0.5, nm.max()+1.5), color=OI['green'], edgecolor='black', linewidth=0.3, alpha=0.85)
    ax[1,0].axvline(2.5, color='gray', ls=':', lw=0.6); ax[1,0].set_xlabel('n distinct metals'); ax[1,0].set_ylabel('count')
    ax[1,1].hist(d.expensive.dropna(), bins=np.linspace(0,1,30), color=OI['orange'], edgecolor='black', linewidth=0.3, alpha=0.85)
    ax[1,1].set_xlabel('expensive-metal fraction (PGM+heavy REE)'); ax[1,1].set_ylabel('count')
    ax[1,2].hexbin(d.co2rr_sel, d.best_co, gridsize=35, mincnt=1, cmap='PuRd', norm=LogNorm())
    ax[1,2].axhline(0, color='gray', ls=':', lw=0.6); ax[1,2].set_xlabel('CO2RR selectivity'); ax[1,2].set_ylabel(r'best-site $\Delta G_{CO}$ (eV)')
fig.tight_layout()
for ext in ('pdf','png'):
    out = FIG/f'fig_co2rr_generation_stats.{ext}'; fig.savefig(out); print('saved', out)
plt.show()


## 4. Guidance-weight comparison (w=1.0 vs w=2.5)

Does stronger guidance raise the yield of economic, CO-releasing HEC? Pass = carbide &
`|best_co|<0.20` & `S_mix>=1.0` & `n_metal>=3` & `expensive==0`.


In [ ]:
def pass_stats(df):
    n = len(df)
    if n == 0: return None
    c = df[df.is_carbide]
    kept = c[(c.best_co.abs() < CO_WINDOW) & (c.S_mix >= 1.0) & (c.n_metal >= 3) & (c.expensive == 0)]
    return {'generated': n, 'carbides': len(c), 'kept_econHEC': len(kept), 'kept_%': round(100*len(kept)/n,2),
            'ge4_metal': int((kept.n_metal >= 4).sum()), 'mean_S_kept': round(kept.S_mix.mean(),3) if len(kept) else np.nan}
cmp = {tag: pass_stats(df) for tag, df in pools.items()}
cmp_df = pd.DataFrame({k: v for k, v in cmp.items() if v}).T
print(cmp_df.to_string())


## 5. Cheap-screen funnel and top economic-HEC candidates

The discovery funnel: generated -> carbide -> economic CO-releasing HEC (cheap screen,
`top_CO.csv`). Top candidates are ranked by closeness to the CO optimum then CO-selective fraction.


In [ ]:
# Funnel counts
n_gen = len(df_all); n_carb = int(df_all.is_carbide.sum()) if len(df_all) else 0
top_csv = GEN/'top_CO.csv'
n_cheap = 0
if top_csv.exists():
    tc = pd.read_csv(top_csv); n_cheap = len(tc)
    n_ucheap = tc['formula'].nunique()
else:
    tc = pd.DataFrame(); n_ucheap = 0
print('=== CO2RR discovery funnel ===')
print(f'  generated structures : {n_gen}')
print(f'  TM carbides          : {n_carb}')
print(f'  cheap-screen survivors (economic CO-releasing HEC): {n_cheap}  ({n_ucheap} unique formulas)')

# Funnel bar figure
if n_gen:
    fig, axf = plt.subplots(figsize=(3.4, 2.2))
    stages = ['generated','carbide','econ-HEC\n(cheap)']; vals = [n_gen, n_carb, n_cheap]
    axf.bar(stages, vals, color=[OI['sky'], OI['blue'], OI['vermil']])
    for i, v in enumerate(vals): axf.text(i, v, str(v), ha='center', va='bottom', fontsize=7)
    axf.set_ylabel('count'); axf.set_title('CO2RR discovery funnel'); axf.set_yscale('log')
    fig.savefig(FIG/'fig_co2rr_funnel.pdf'); fig.savefig(FIG/'fig_co2rr_funnel.png'); print('saved', FIG/'fig_co2rr_funnel.png')
    plt.show()

# Top economic-HEC candidates from the cheap screen
if len(tc):
    cols = [c for c in ['formula','best_site_co_affinity','frac_co_selective_sites','mean_cooh_affinity','co2rr_selectivity','metal_mixing_entropy','n_metal_species','expensive_metal_fraction'] if c in tc.columns]
    print('\n=== top cheap-screen economic-HEC candidates ==='); print(tc[cols].head(15).to_string(index=True))
    from pymatgen.core import Composition; from collections import Counter
    cnt = Counter()
    for f in tc['formula']:
        for el in Composition(f).elements:
            if el.symbol != 'C': cnt[el.symbol] += 1
    print('\nmost common metals among survivors:', cnt.most_common(15))


## 6. Takeaways

- Stage A generates a large carbide pool; the cheap screen funnels it to economic,
  CO-releasing high-entropy carbides (PGM/heavy-REE-free).
- Survivors are dominated by cheap early-TM + light-REE metals (Y, Zr, La, Ce, Pr, W).
- The fairchem surface screen (notebook 18) then ranks these on real $\Delta G_{CO}$,
  and QE (notebook 20) validates the top 6 for bulk stability + metallicity.

_Re-run to refresh as generation / cheap-screen outputs change._
